# Stage 4 — Small Reasoning Probe: Molmo2 vs Qwen3-VL vs InternVL3

Self-contained (no `git clone`/`src` import).

**What this tests:** not symbol detection — a short counting/comparison question over
the image ("which symbol type is most common here, and roughly how many?"), checked
against real ground-truth counts from the Kaggle labels. This is a small, indicative
signal about general reasoning/instruction-following, not a rigorous benchmark — one
prompt, a handful of tiles, judged by eye against the real answer.

**Why this test specifically:** Molmo2 is trained heavily around pointing-style output;
Qwen3-VL/InternVL3 are general-purpose VLMs more used to open-ended Q&A. If Molmo2
struggles to give a coherent counting/comparison answer while the other two manage it,
that's a real (if small) data point in the direction of the "unproven on reasoning"
caveat already in CLAUDE.md — not proof, an indication.

One model is loaded, tested, and freed before the next loads, to fit on one GPU.

## 1. Mount Drive, check GPU

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader

MessageError: User cancelled dfs_ephemeral authorization

## 2. Pick 3 labeled Kaggle tiles + compute real ground-truth majority class

(No model loaded yet — just picking tiles and reading their real labels, so we know
the correct answer before asking anything.)

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/pid_project/data"
KAGGLE_DIR = "kaggle_pid_symbols"

import json, random
from collections import Counter
from pathlib import Path
from PIL import Image

ROOT = Path(DRIVE_ROOT)
kaggle_p = ROOT / KAGGLE_DIR

classes_path = ROOT / "classes.json"
class_names = {}
if classes_path.exists():
    data = json.loads(classes_path.read_text())
    class_names = {cid: entry["kaggle_name"] for cid, entry in data["classes"].items()}

random.seed(1)
candidates = []
for lbl in (kaggle_p / "labels").glob("*.txt"):
    lines = [l for l in lbl.read_text().splitlines() if l.strip()]
    if len(lines) < 6:
        continue
    counts = Counter(l.split()[0] for l in lines)
    top_cls, top_n = counts.most_common(1)[0]
    # only keep tiles with an unambiguous majority (not a 3-way tie etc.)
    if top_n >= 3 and (len(counts) == 1 or top_n > counts.most_common(2)[1][1]):
        candidates.append((lbl, top_cls, top_n, len(lines)))
    if len(candidates) >= 200:
        break

probe_tiles = random.sample(candidates, 3)
for lbl, top_cls, top_n, total in probe_tiles:
    name = class_names.get(top_cls, f"class_{top_cls}")
    print(f"{lbl.stem:20s} majority={name:20s} count={top_n}  (total boxes in tile: {total})")

## 3. The reasoning prompt (same question, all 3 candidates)

In [ ]:
REASONING_PROMPT = (
    "Look at this P&ID tile. Which type of symbol appears most frequently in this image, "
    "and approximately how many of that symbol are there? Answer in one or two sentences."
)

def show_probe_images():
    for lbl, top_cls, top_n, total in probe_tiles:
        img_path = kaggle_p / "images" / f"{lbl.stem}.jpg"
        print(f"--- {lbl.stem} --- ground truth: {class_names.get(top_cls, top_cls)} x{top_n} (of {total} total boxes)")
        display(Image.open(img_path))

## 4. Qwen3-VL

In [ ]:
!pip install -q transformers accelerate qwen-vl-utils

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

qwen_processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-8B-Instruct")
qwen_model = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen3-VL-8B-Instruct", dtype=torch.bfloat16, device_map="cuda"
)
print("Qwen3-VL loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

for lbl, top_cls, top_n, total in probe_tiles:
    img = Image.open(kaggle_p / "images" / f"{lbl.stem}.jpg").convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": REASONING_PROMPT}]}]
    inputs = qwen_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
    ).to(qwen_model.device)
    out = qwen_model.generate(**inputs, max_new_tokens=150, do_sample=False)
    answer = qwen_processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    print(f"\n=== {lbl.stem} (ground truth: {class_names.get(top_cls, top_cls)} x{top_n}) ===")
    print("Qwen3-VL:", answer.strip())

del qwen_model, qwen_processor
torch.cuda.empty_cache()
print("\nFreed Qwen3-VL from GPU.")

## 5. InternVL3

In [ ]:
internvl_processor = AutoProcessor.from_pretrained("OpenGVLab/InternVL3-8B-hf")
internvl_model = AutoModelForImageTextToText.from_pretrained(
    "OpenGVLab/InternVL3-8B-hf", dtype=torch.bfloat16, device_map="cuda"
)
print("InternVL3 loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

for lbl, top_cls, top_n, total in probe_tiles:
    img = Image.open(kaggle_p / "images" / f"{lbl.stem}.jpg").convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": REASONING_PROMPT}]}]
    inputs = internvl_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
    ).to(internvl_model.device)
    out = internvl_model.generate(**inputs, max_new_tokens=150, do_sample=False)
    answer = internvl_processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    print(f"\n=== {lbl.stem} (ground truth: {class_names.get(top_cls, top_cls)} x{top_n}) ===")
    print("InternVL3:", answer.strip())

del internvl_model, internvl_processor
torch.cuda.empty_cache()
print("\nFreed InternVL3 from GPU.")

## 6. Molmo2-O-7B

Requires `transformers==4.57.1` — **restart the runtime after this install cell**, then
re-run cells 2-3 (tile selection) before continuing to the load+probe cell below.

In [ ]:
!pip install -q transformers==4.57.1 accelerate

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

molmo_processor = AutoProcessor.from_pretrained("allenai/Molmo2-O-7B", trust_remote_code=True, dtype="auto")
molmo_model = AutoModelForImageTextToText.from_pretrained(
    "allenai/Molmo2-O-7B", trust_remote_code=True, dtype="auto", device_map="cuda"
)
print("Molmo2 loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

for lbl, top_cls, top_n, total in probe_tiles:
    img = Image.open(kaggle_p / "images" / f"{lbl.stem}.jpg").convert("RGB")
    messages = [{"role": "user", "content": [{"type": "text", "text": REASONING_PROMPT}, {"type": "image", "image": img}]}]
    inputs = molmo_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
    ).to(molmo_model.device)
    out = molmo_model.generate(**inputs, max_new_tokens=150, do_sample=False)
    answer = molmo_processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    print(f"\n=== {lbl.stem} (ground truth: {class_names.get(top_cls, top_cls)} x{top_n}) ===")
    print("Molmo2:", answer.strip())

## 7. What to look for

For each tile, compare the three answers against the printed ground truth:
- Does the answer name a plausible symbol type, or is it garbled/off-topic/just points?
- Is the count in the right ballpark (doesn't need to be exact)?
- Does it actually answer the comparison question, or does it just describe the image generically?

Paste the three candidates' raw answers back — this determines what (if anything) goes
into the write-up. A model doing noticeably worse here isn't proof of anything on its own,
but it's a real, checkable data point, not an assumption.